# 01 · Fine-tune the NER model (TPU or GPU)

Trains a token classifier for drawing entities (MATERIAL, COATING, PART_NUMBER, STANDARD, THICKNESS, DIMENSION)
and logbook entities. It replaces the gazetteer in layer 2 of the extraction cascade.

**Runtime:** *Runtime → Change runtime type → TPU* (v5e/v6e). The cell below installs PyTorch/XLA so the Hugging Face
Trainer runs on the TPU. If the TPU image refuses the install, switch to a **T4 GPU** runtime — every other cell is
unchanged. A DistilRoBERTa NER model trains in a few minutes on either.

**Data:** the repo's BIO-tagged synthetic drawings (license-free) + optionally the MaintNet aviation logbook set
(COLING 2020, open source) for abbreviation-heavy maintenance text.

In [ ]:
# Get the repo, regenerate the synthetic corpus (deterministic, ~5 s) and fetch the public datasets (checksummed)
!git clone -q https://github.com/<you>/aeronexus-ai.git /content/aeronexus-ai || true
%cd /content/aeronexus-ai/backend
!pip -q install -r requirements.txt
!python scripts/generate_data.py --train 4000
!python scripts/fetch_public_data.py

In [ ]:
# TPU support for PyTorch (skip on GPU). Versions must match the preinstalled torch.
import importlib, subprocess, sys
try:
    import jax; tpu = any(d.platform == "tpu" for d in jax.devices())
except Exception:
    tpu = False
if tpu:
    import torch
    v = torch.__version__.split("+")[0]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"torch_xla[tpu]=={v}",
                    "-f", "https://storage.googleapis.com/libtpu-releases/index.html"], check=False)
!pip -q install transformers datasets accelerate seqeval evaluate
print("TPU runtime:", tpu)

In [ ]:
import json, random
from datasets import Dataset, DatasetDict

rows = [json.loads(l) for l in open("data/train/ner_bio.jsonl")]
random.Random(0).shuffle(rows)
labels = sorted({t for r in rows for t in r["tags"]}, key=lambda t: (t != "O", t))
label2id = {l: i for i, l in enumerate(labels)}; id2label = {i: l for l, i in label2id.items()}
n = len(rows)
ds = DatasetDict({
    "train": Dataset.from_list(rows[: int(.8 * n)]),
    "validation": Dataset.from_list(rows[int(.8 * n): int(.9 * n)]),
    "test": Dataset.from_list(rows[int(.9 * n):]),
})
print(ds, labels)

In [ ]:
# (optional) add MaintNet aviation logbook text for domain robustness.
# Download the aviation CSV from the MaintNet repository (see docs/DATASETS.md), upload it, then weak-label it with
# the repo's gazetteer so the model learns abbreviations/misspellings it has never seen in drawings.
import os
if os.path.exists("/content/maintnet_aviation.csv"):
    import pandas as pd, sys
    sys.path.insert(0, ".")
    from aeronexus.extraction.ner import GazetteerNER
    g = GazetteerNER()
    df = pd.read_csv("/content/maintnet_aviation.csv")
    text_col = next(c for c in df.columns if df[c].dtype == object)
    extra = []
    for txt in df[text_col].dropna().astype(str).head(5000):
        words = txt.split(); tags = ["O"] * len(words); pos = 0
        for i, w in enumerate(words):
            start = txt.find(w, pos); pos = start + len(w)
            for e in g(txt):
                if e.start <= start < e.end and e.label in {"COMPONENT", "ACTION"}:
                    tags[i] = ("B-" if start == e.start else "I-") + e.label
        extra.append({"id": f"mn{len(extra)}", "tokens": words, "tags": tags})
    for l in {t for r in extra for t in r["tags"]} - set(label2id):
        label2id[l] = len(label2id); id2label[label2id[l]] = l
    from datasets import concatenate_datasets
    ds["train"] = concatenate_datasets([ds["train"], Dataset.from_list(extra)])
    print("added MaintNet rows:", len(extra))

In [ ]:
from transformers import AutoTokenizer
MODEL = "distilroberta-base"
tok = AutoTokenizer.from_pretrained(MODEL, add_prefix_space=True)

def encode(batch):
    enc = tok(batch["tokens"], is_split_into_words=True, truncation=True, max_length=256)
    out = []
    for i, tags in enumerate(batch["tags"]):
        prev, lab = None, []
        for wid in enc.word_ids(i):
            lab.append(-100 if wid is None or wid == prev else label2id[tags[wid]])  # label first sub-token only
            prev = wid
        out.append(lab)
    enc["labels"] = out
    return enc

tokenized = ds.map(encode, batched=True, remove_columns=ds["train"].column_names)

In [ ]:
import numpy as np, evaluate
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification, Trainer, TrainingArguments
seqeval = evaluate.load("seqeval")
model = AutoModelForTokenClassification.from_pretrained(MODEL, num_labels=len(label2id), id2label=id2label, label2id=label2id)

def metrics(p):
    preds, labs = np.argmax(p.predictions, -1), p.label_ids
    P = [[id2label[x] for x, y in zip(pr, la) if y != -100] for pr, la in zip(preds, labs)]
    L = [[id2label[y] for x, y in zip(pr, la) if y != -100] for pr, la in zip(preds, labs)]
    r = seqeval.compute(predictions=P, references=L)
    return {"precision": r["overall_precision"], "recall": r["overall_recall"], "f1": r["overall_f1"]}

args = TrainingArguments("ner-out", learning_rate=5e-5, per_device_train_batch_size=32, per_device_eval_batch_size=64,
                         num_train_epochs=3, weight_decay=0.01, eval_strategy="epoch", save_strategy="no",
                         logging_steps=50, report_to=[], seed=7)
trainer = Trainer(model=model, args=args, train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],
                  data_collator=DataCollatorForTokenClassification(tok), compute_metrics=metrics)
trainer.train()
test_metrics = trainer.evaluate(tokenized["test"]); print(test_metrics)

In [ ]:
# Export in the exact layout the backend loads (backend/models/ner) + a model card
out = "/content/export/ner"
trainer.save_model(out); tok.save_pretrained(out)
json.dump({"base_model": MODEL, "labels": labels, "test": test_metrics,
           "data": "AeroNexus synthetic drawings (+MaintNet if provided)", "intended_use": "entity tagging in drawings/logbooks",
           "limitations": "synthetic layouts; verify on your own labelled drawings before relying on it"},
          open(f"{out}/model_card.json", "w"), indent=1)
ARTIFACTS = ["/content/export/ner"]

In [ ]:
# Save artifacts to Google Drive (survives the Colab session; avoids flaky browser downloads of big files).
# In VS Code: python scripts/import_models.py <the downloaded aeronexus_export.zip or Drive folder>
import hashlib, json, os, platform, shutil, sys, time
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/aeronexus/export"
os.makedirs(OUT, exist_ok=True)
for item in ARTIFACTS:                                   # files or folders produced above
    dst = os.path.join(OUT, os.path.basename(item.rstrip("/")))
    (shutil.copytree(item, dst, dirs_exist_ok=True) if os.path.isdir(item) else shutil.copy2(item, dst))
def sha(p):
    h = hashlib.sha256(); h.update(open(p, "rb").read()); return h.hexdigest()
import sklearn, numpy
man_path = os.path.join(OUT, "manifest.json")
man = json.load(open(man_path)) if os.path.exists(man_path) else {"artifacts": {}}
for root, _, fs in os.walk(OUT):
    for f in fs:
        if f != "manifest.json":
            p = os.path.join(root, f); man["artifacts"][os.path.relpath(p, OUT)] = sha(p)
man.update({"exported_at": time.strftime("%Y-%m-%dT%H:%M:%S"), "python": platform.python_version(),
            "versions": {"scikit-learn": sklearn.__version__, "numpy": numpy.__version__}})
json.dump(man, open(man_path, "w"), indent=1)
shutil.make_archive("/content/drive/MyDrive/aeronexus/aeronexus_export", "zip", OUT)
print("saved to Drive: MyDrive/aeronexus/aeronexus_export.zip  →", sorted(man["artifacts"])[:12])

**Next (VS Code):** unzip into `backend/models/ner/`, then run with the trained model:

```bash
cd backend
AERONEXUS_USE_HF_NER=1 python -m aeronexus.evals.runner   # gate must still pass
AERONEXUS_USE_HF_NER=1 pytest -q
```